In [3]:
!jupyter nbconvert --to notebook --execute preprocess.ipynb --inplace

[NbConvertApp] Converting notebook preprocess.ipynb to notebook
[NbConvertApp] Writing 122156 bytes to preprocess.ipynb


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from pulp import *
from scipy.stats import poisson, norm
from pulp import LpProblem, LpMaximize, LpVariable, LpStatus, lpSum, value, PULP_CBC_CMD

In [3]:
OUT_CSV = "fantasy_enriched.csv"



| Position        | Point Source         | Points | Description                                                                 |
| --------------- | -------------------- | -----: | --------------------------------------------------------------------------- |
| **All Players** | Appearance (<60 min) |     +1 | Plays any minutes up to 59:59                                               |
| **All Players** | Appearance (60+ min) |     +1 | Additional point for reaching 60+ minutes (total appearance = **2 points**) |
| **All Players** | Assist               |     +3 | Provides an assist                                                          |
| **All Players** | Yellow Card          |     -1 | Receives a yellow card                                                      |
| **All Players** | Red Card             |     -2 | Receives a red card                                                         |
| **All Players** | Own Goal             |     -2 | Scores an own goal                                                          |
| **All Players** | Winning a Penalty    |     +2 | Wins a penalty for their team                                               |
| **All Players** | Conceding a Penalty  |     -1 | Commits a foul leading to a penalty                                         |

| Position       | Point Source                  | Points | Description                                        |
| -------------- | ----------------------------- | -----: | -------------------------------------------------- |
| **Goalkeeper** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet     |
| **Goalkeeper** | First Goal Conceded           |      0 | No deduction for conceding the first goal          |
| **Goalkeeper** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first                |
| **Goalkeeper** | Goal Scored                   |     +9 | Scores a goal                                      |
| **Goalkeeper** | Penalty Save                  |     +3 | Saves a penalty during normal play (not shootouts) |
| **Goalkeeper** | Every 3 Saves                 |     +1 | Earns 1 point for every 3 saves                    |

| Position     | Point Source                  | Points | Description                                    |
| ------------ | ----------------------------- | -----: | ---------------------------------------------- |
| **Defender** | Clean Sheet                   |     +5 | Plays 60+ minutes and team keeps a clean sheet |
| **Defender** | First Goal Conceded           |      0 | No deduction for conceding the first goal      |
| **Defender** | Each Additional Goal Conceded |     -1 | Every goal conceded after the first            |
| **Defender** | Goal Scored                   |     +7 | Scores a goal                                  |

| Position       | Point Source                | Points | Description                                    |
| -------------- | --------------------------- | -----: | ---------------------------------------------- |
| **Midfielder** | Clean Sheet                 |     +1 | Plays 60+ minutes and team keeps a clean sheet |
| **Midfielder** | Goal Scored                 |     +6 | Scores a goal                                  |
| **Midfielder** | Every 3 Tackles             |     +1 | Earns 1 point for every 3 tackles              |
| **Midfielder** | Every 2 Big Chances Created |     +1 | Earns 1 point for every 2 big chances created  |

| Position    | Point Source            | Points | Description                               |
| ----------- | ----------------------- | -----: | ----------------------------------------- |
| **Forward** | Goal Scored             |     +5 | Scores a goal                             |
| **Forward** | Every 2 Shots on Target |     +1 | Earns 1 point for every 2 shots on target |

| Position                | Point Source          | Points | Description                                                                                                                                                                        |
| ----------------------- | --------------------- | -----: | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Bonus (All Players)** | Direct Free-Kick Goal |     +1 | Additional bonus for scoring directly from a free kick (on top of goal points)                                                                                                     |
| **Bonus (All Players)** | Scouting Bonus        |     +2 | Scores **more than 4 base fantasy points** in a match **and** is selected by **<5%** of fantasy teams. Captaincy and booster points do **not** count toward the 4-point threshold. |



## Core identity / player info

| Column | Type | Description |
|---|---|---|
| `fifa_id` | int | Unique FIFA fantasy player ID |
| `name` | str | Player name |
| `squad_id` | int | Internal squad/nation ID |
| `team` | str | National team |
| `position` | str | DEF / MID / FWD / GK |
| `price` | float | Fantasy price |
| `status` | str | Availability status |
| `total_points` | int | Total tournament points |
| `avg_points` | float | Average points per match |
| `matches_played` | int | Matches with participation |
| `percent_selected` | float | Selection rate (%) |
| `next_fixture` | int | Next match ID |


## Round fantasy points

| Column | Type | Description |
|---|---|---|
| `round_1_points` | int | Points in round 1 |
| `round_2_points` | int | Points in round 2 |
| `round_3_points` | int | Points in round 3 |
| `round_4_points` | int | Points in round 4 |


## Betting / scorer mapping

| Column | Type | Description |
|---|---|---|
| `betano_matched_name` | str | Matched Betano market name |
| `betano_match_score` | float | Name matching confidence (0–100) |
| `anytime_scorer_prob` | float | Probability player scores anytime |
| `first_scorer_prob` | float | Probability scores first goal |
| `last_scorer_prob` | float | Probability scores last goal |


## Match context

| Column | Type | Description |
|---|---|---|
| `match_home` | str | Home team |
| `match_away` | str | Away team |
| `is_home` | bool | Player team is home |
| `opponent` | str | Opponent team |


## Match probabilities 

| Column | Type | Description |
|---|---|---|
| `match_home_win_prob` | float | Home win probability |
| `match_draw_prob` | float | Draw probability |
| `match_away_win_prob` | float | Away win probability |
| `match_btts_prob` | float | Both teams score probability |
| `match_over_25_prob` | float | Over 2.5 goals probability |
| `match_over_35_prob` | float | Over 3.5 goals probability |
| `team_score_prob` | float | Team scores ≥1 goal probability |
| `team_score_2_prob` | float | Team scores ≥2 goals probability |
| `team_cs_prob` | float | Team clean sheet probability |
| `opp_score_prob` | float | Opponent scores ≥1 goal probability |
| `opp_cs_prob` | float | Opponent clean sheet probability |
| `team_over_05_prob` | float | Team scores ≥1 goal probability |
| `team_over_15_prob` | float | Team scores ≥2 goals probability |
| `team_qualify_prob` | float | Team advances probability |
| `team_win2_prob` | float | Team wins by 2+ goals probability |
| `opp_over_05_prob` | float | Opponent scores ≥1 goal probability |


## Raw cumulative stats

| Column | Type | Description |
|---|---|---|
| `stat_GS` | int | Goals |
| `stat_AS` | int | Assists |
| `stat_CS` | int | Clean sheets |
| `stat_GC` | int | Goals conceded |
| `stat_MP` | int | Minutes played |
| `stat_YC` | int | Yellow cards |
| `stat_RC` | int | Red cards |
| `stat_ST` | int | Shots on target |
| `stat_SB` | int | Shots blocked |
| `stat_CC` | int | Chances created |
| `stat_PS` | int | Penalties saved |
| `stat_T` | int | Tackles |
| `stat_S` | int | Saves |
| `stat_SXI` | int | Starts (XI appearances) |
| `stat_OG` | int | Own goals |
| `stat_PC` | int | Penalties committed |
| `stat_PW` | int | Penalties won |
| `stat_FK` | int | Free kicks won |



## Round-by-round stats (1–4)

| Pattern | Type | Description |
|---|---|---|
| `round_i_SXI` | int | Started (0/1) |
| `round_i_MP` | int | Minutes |
| `round_i_AS` | int | Assists |
| `round_i_YC` | int | Yellow cards |
| `round_i_RC` | int | Red cards |
| `round_i_OG` | int | Own goals |
| `round_i_PW` | int | Penalties won |
| `round_i_PC` | int | Penalties committed |
| `round_i_CS` | int | Clean sheets |
| `round_i_GS` | int | Goals |
| `round_i_GC` | int | Goals conceded |
| `round_i_PS` | int | Penalties saved |
| `round_i_T` | int | Tackles |
| `round_i_CC` | int | Chances created |
| `round_i_ST` | int | Shots on target |
| `round_i_FK` | int | Free kicks |
| `round_i_S` | int | Saves |
| `round_i_SB` | int | Shots blocked |



## External matching / enrichment

| Column | Type | Description |
|---|---|---|
| `clubstats_matched_player` | str | External player match |
| `clubs_match_dist` | float | Matching distance score |



## Form / usage trends

| Column | Type | Description |
|---|---|---|
| `started_last_match` | int | Started last match |
| `starts_last_3` | int | Starts last 3 matches |
| `starts_last_5` | int | Starts last 5 matches |
| `start_rate` | float | Start frequency |
| `minutes_last_3_avg` | float | Avg minutes last 3 |
| `minutes_last_5_avg` | float | Avg minutes last 5 |



## Performance rates

| Column | Type | Description |
|---|---|---|
| `goal_rate` | float | Goals per minute |
| `goals_per_start` | float | Goals per start |
| `shots_on_target_per90` | float | SOT per 90 |
| `goals_per_shot_on_target` | float | Conversion rate |
| `assist_rate` | float | Assists per minute |
| `chance_created_per90` | float | Chances per 90 |
| `tackles_per90` | float | Tackles per 90 |
| `cc_per90` | float | Chances created per 90 |
| `sot_per90` | float | Shots on target per 90 |
| `saves_per90` | float | Saves per 90 |
| `yc_per90` | float | Yellow cards per 90 |
| `rc_per90` | float | Red cards per 90 |
| `penalty_conceded_rate` | float | Penalties conceded per 90 |
| `own_goal_rate` | float | Own goals per 90 |
| `penalties_won_per90` | float | Penalties won per 90 |



## Recent / streak features

| Column | Type | Description |
|---|---|---|
| `recent_goals_last3` | int | Goals last 3 matches |
| `goal_streak` | int | Consecutive scoring matches |
| `recent_assists_last3` | int | Assists last 3 matches |
| `recent_chances_created_per90` | float | Recent creation rate |
| `recent_cs_rate` | float | Clean sheet rate |
| `recent_tackles_per90` | float | Recent tackles |
| `recent_cc_per90` | float | Recent chances created |
| `recent_sot_per90` | float | Recent shots on target |
| `recent_saves_per90` | float | Recent saves |
| `recent_penalties_won` | int | Penalties won last 3 |



## Expected value features

| Column | Type | Description |
|---|---|---|
| `goal_expectation` | float | Scoring proxy |
| `goal_expectation2` | float | Scoring proxy (2+ goals) |
| `assist_expectation` | float | Assist proxy |
| `cs_expectation` | float | Clean sheet proxy |
| `expected_minutes` | float | Expected minutes |
| `expected_gc_penalty` | float | Defensive penalty |
| `expected_tackle_points` | float | Tackles contribution |
| `expected_cc_points` | float | Chance creation points |
| `expected_sot_points` | float | Shot contribution points |
| `expected_save_points` | float | Save contribution points |



## Fantasy points dynamics

| Column | Type | Description |
|---|---|---|
| `points_last1` | int | Last match points |
| `points_last3_avg` | float | Avg last 3 |
| `points_last5_avg` | float | Avg last 5 |
| `weighted_points` | float | Recency weighted |
| `rolling_std_points` | float | Variance last 5 |
| `rolling_max_points` | int | Max last 5 |



## Team-position ranking features

| Column | Type | Description |
|---|---|---|
| `price_rank_team_position` | float | Price rank |
| `selected_rank_team_position` | float | Selection rank |
| `points_rank_team_position` | float | Points rank |
| `minutes_rank_team_position` | float | Minutes rank |
| `starts_rank_team_position` | float | Starts rank |
| `anytime_rank_team_position` | float | Scoring odds rank |
| `shots_rank_team_position` | float | Shooting rank |
| `cc_rank_team_position` | float | Chance creation rank |
| `tackles_rank_team_position` | float | Tackles rank |

In [4]:
df= pd.read_csv("fantasy_enriched.csv")

In [5]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [6]:
df = df[df["status"] == "playing"].copy()

In [7]:
df.loc[df["name"] == "Johan Manzambi", "status"] = "injured"  # :(

In [51]:
# probability of playing a minute

raw_any = (
    0.2* df["minutes_rank_team_position"]
  + 0.10 * df["stat_MP"]        # minutes needs to be very team stratified since while for other point sources players are all eligible
  + 0.15 * df["starts_rank_team_position"]
  + 0.10 * df["price_rank_team_position"]     # for minutes its a team stratified signal.
  + 0.1 * df["selected_rank_team_position"]
  + 0.25 * df["minutes_last_3_avg"]
  + 0.1 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_any"] = sigmoid((raw_any - 0.5) * 6) 

In [52]:
df.loc[
    :,
    [
        "prob_plays_any",
        "name",
        "team",
        "position",
        "stat_MP",
        "minutes_rank_team_position",
        "starts_rank_team_position",
        "selected_rank_team_position",
        "price_rank_team_position",
        "anytime_rank_team_position",
    ],
].sort_values("prob_plays_any", ascending=False).head(30)

,prob_plays_any,name,team,position,stat_MP,minutes_rank_team_position,starts_rank_team_position,selected_rank_team_position,price_rank_team_position,anytime_rank_team_position
192,0.931263,Achraf Hakimi,Morocco,DEF,1.000000,1.000000,0.562500,1.000000,1.000000,1.000000
68,0.921380,Kylian Mbappé,France,FWD,0.918750,1.000000,0.666667,1.000000,1.000000,1.000000
47,0.919179,Harry Kane,England,FWD,0.922917,1.000000,0.625000,1.000000,1.000000,1.000000
118,0.910926,Mikel Oyarzabal,Spain,FWD,0.812500,1.000000,0.625000,1.000000,1.000000,1.000000
39,0.901679,Youri Tielemans,Belgium,MID,0.989583,1.000000,0.545455,0.700000,0.727273,0.590909
164,0.898915,Leandro Trossard,Belgium,MID,0.937500,0.909091,0.545455,1.000000,0.818182,0.863636
63,0.895824,Dayot Upamecano,France,DEF,0.908333,1.000000,0.555556,0.877778,0.833333,0.888889
81,0.894151,Michael Olise,France,MID,0.829167,1.000000,0.545455,1.000000,0.909091,0.681818
44,0.894022,Ezri Konsa,England,DEF,0.937500,1.000000,0.555556,0.755556,0.777778,0.500000
114,0.892712,Marc Cucurella,Spain,DEF,0.937500,0.937500,0.562500,1.000000,0.625000,0.500000


In [57]:
# probability of starting / playing 60 minutes

raw_any = (
    0.25 * df["minutes_rank_team_position"]
  + 0.05 * df["starts_rank_team_position"]
  + 0.20 * df["stat_MP"]
  + 0.05 * df["selected_rank_team_position"]
  + 0.40 * df["minutes_last_3_avg"]
  + 0.05 * df["has_betano_match"] # having odd at all means likeliness of playing is higher.

)

df["prob_plays_60min"] = sigmoid((raw_any - 0.5) * 6)


In [59]:
cols = [
    "prob_plays_60min",
    "name",
    "team",
    "position",
    "minutes_rank_team_position",
    "starts_rank_team_position",
    "price_rank_team_position",
    "selected_rank_team_position",
    "minutes_last_3_avg",
    "round_5_MP",
    "anytime_rank_team_position",
]

df[cols].sort_values("prob_plays_60min", ascending=False).head(30)

,prob_plays_60min,name,team,position,minutes_rank_team_position,starts_rank_team_position,price_rank_team_position,selected_rank_team_position,minutes_last_3_avg,round_5_MP,anytime_rank_team_position
192,0.946280,Achraf Hakimi,Morocco,DEF,1.000000,0.562500,1.000000,1.000000,1.000000,0.750000,1.000000
28,0.943303,Brandon Mechele,Belgium,DEF,1.000000,0.555556,0.277778,0.816667,1.000000,0.750000,0.722222
39,0.938262,Youri Tielemans,Belgium,MID,1.000000,0.545455,0.727273,0.700000,0.983333,0.750000,0.590909
199,0.934919,Neil El Aynaoui,Morocco,MID,1.000000,0.555556,0.444444,0.572222,0.976667,0.750000,0.666667
10,0.930862,Emiliano Martínez,Argentina,GK,1.000000,0.666667,1.000000,1.000000,1.000000,0.750000,0.666667
160,0.930862,Thibaut Courtois,Belgium,GK,1.000000,0.666667,1.000000,1.000000,1.000000,0.750000,0.666667
144,0.930862,Gregor Kobel,Switzerland,GK,1.000000,0.666667,1.000000,1.000000,1.000000,1.000000,0.666667
191,0.930862,Yassine Bounou,Morocco,GK,1.000000,0.666667,1.000000,1.000000,1.000000,0.750000,0.666667
128,0.926355,Granit Xhaka,Switzerland,MID,1.000000,0.550000,0.850000,0.890000,1.000000,1.000000,0.550000
47,0.924072,Harry Kane,England,FWD,1.000000,0.625000,1.000000,1.000000,0.876667,0.741667,1.000000


There is a phenomenon where a player will be overvalued, such as Lukaku here for belgium. He has not played much, but since Belgium only has 3 forward and only him has significative minutes he ranks number 1 for all ranks in BEL,FWD. Need to include more general sums and beware with manual selection.

In [12]:
df["lambda_goal"] = -np.log(
    (1 - df["anytime_scorer_prob"]).clip(lower=0.001)
)

lambda_adj = (
    0.70 * df["lambda_goal"]
  + 0.1 * df["recent_goals_last3"]
  + 0.02 * df["team_score_2_prob"]
  + 0.1 * df["stat_GS"]
  + 0.05 * df["stat_ST"]
  + 0.03 * df["price"]
)

df["prob_scores"] = (
    lambda_adj.clip(lower=0)
    * df["prob_plays_60min"]
    * 1.5
)

df.loc[df["position"] == "GK", "prob_scores"] = 0.
df.loc[df["position"] == "DEF", "prob_scores"] = df["prob_scores"] * 0.7

In [13]:
cols = [
    "name",
    "team",
    "position",
    "prob_scores",
    "goal_expectation",
    "anytime_scorer_prob",
    "recent_goals_last3",
    "recent_sot_per90",
    "goal_rate",
    "shots_on_target_per90",
    "stat_GS",
    "stat_ST",
    "price_rank_team_position",
]

df[cols].sort_values("prob_scores", ascending=False).head(30)

,name,team,position,prob_scores,goal_expectation,anytime_scorer_prob,recent_goals_last3,recent_sot_per90,goal_rate,shots_on_target_per90,stat_GS,stat_ST,price_rank_team_position
68,Kylian Mbappé,France,FWD,1.181813,0.463796,0.5882,0.75,0.725191,0.813492,0.616780,0.875,1.000000,1.000000
47,Harry Kane,England,FWD,1.100961,0.433924,0.5556,1.00,0.433460,0.694131,0.361174,0.750,0.588235,1.000000
118,Mikel Oyarzabal,Spain,FWD,0.922928,0.425540,0.5348,0.50,0.372549,0.525641,0.369231,0.500,0.529412,1.000000
99,Erling Haaland,Norway,FWD,0.890283,0.332962,0.5051,0.75,0.527778,0.996528,0.533333,0.875,0.705882,1.000000
69,Ousmane Dembélé,France,MID,0.666236,0.315400,0.4000,0.75,0.254464,0.551075,0.215054,0.500,0.294118,1.000000
60,Jude Bellingham,England,MID,0.619632,0.264759,0.3390,0.75,0.529880,0.507426,0.356436,0.500,0.529412,0.900000
116,Lamine Yamal,Spain,MID,0.600804,0.345970,0.4348,0.00,0.454183,0.162698,0.406349,0.125,0.470588,1.000000
31,Romelu Lukaku,Belgium,FWD,0.402000,0.194641,0.3279,0.75,0.553398,0.768750,0.240000,0.375,0.176471,1.000000
81,Michael Olise,France,MID,0.382240,0.228586,0.2899,0.00,0.237500,0.000000,0.201005,0.000,0.294118,0.909091
164,Leandro Trossard,Belgium,MID,0.368355,0.127683,0.2151,0.50,0.211111,0.227778,0.106667,0.250,0.176471,0.818182


In [14]:
prob_assists = (
    0.5 * df["chance_created_per90"]
  + 0.15 * df["team_score_2_prob"]
  + 0.1 * df["prob_scores"]
  + 0.1 * df["assist_rate"]
  + 0.15 * df["stat_AS"]
)

df["lambda_assist"] = -np.log(
    (1 - prob_assists).clip(lower=0.001)
)

df["expected_assists"] = df["lambda_assist"].clip(lower=0) * df["prob_plays_60min"] 

In [15]:
df.loc[df["position"] == "GK", "expected_assists"] = df["expected_assists"] * 0.1
df.loc[df["position"] == "DEF", "expected_assists"] = df["expected_assists"] * 0.9

In [16]:
cols = [
    "name",
    "team",
    "position",
    "expected_assists",
    "assist_expectation",
    "assist_rate",
    "chance_created_per90",
    "team_score_2_prob",
    "recent_assists_last3",
    "prob_plays_60min",
    "stat_AS",
]

df[cols].sort_values("expected_assists", ascending=False).head(30)

,name,team,position,expected_assists,assist_expectation,assist_rate,chance_created_per90,team_score_2_prob,recent_assists_last3,prob_plays_60min,stat_AS
81,Michael Olise,France,MID,1.158002,0.891520,0.766332,0.766332,0.5882,0.666667,0.868872,1.0
5,Lionel Messi,Argentina,FWD,0.813874,0.996498,0.148780,0.892683,0.5348,0.333333,0.872556,0.2
200,Brahim Díaz,Morocco,MID,0.689002,0.521072,0.628866,0.628866,0.2817,0.666667,0.830569,0.8
90,Patrick Berg,Norway,MID,0.507070,0.549333,0.376543,0.564815,0.3891,0.333333,0.844290,0.4
48,Bukayo Saka,England,MID,0.477514,0.732187,0.953125,0.635417,0.5714,0.666667,0.506125,0.6
60,Jude Bellingham,England,MID,0.474535,0.521955,0.150990,0.452970,0.5714,0.333333,0.873194,0.2
114,Marc Cucurella,Spain,DEF,0.464554,0.477420,0.406667,0.406667,0.6061,0.666667,0.899348,0.6
37,Nicolas Raskin,Belgium,MID,0.429028,0.641088,0.488000,0.732000,0.3077,0.666667,0.569039,0.4
164,Leandro Trossard,Belgium,MID,0.424775,0.356160,0.271111,0.406667,0.3077,0.666667,0.908604,0.4
59,Declan Rice,England,MID,0.415637,0.618387,0.178886,0.536657,0.5714,0.000000,0.764385,0.2


In [17]:
position_yc_modifier = {
    "GK": 0.15,
    "FWD": 0.35,
    "DEF": 0.45,
    "MID": 0.55,
}

df["position_yc_modifier"] = df["position"].map(position_yc_modifier)

raw_yc = (
    0.4 * df["yc_per90"]
  + 0.25 * df["tackles_per90"]
  + 0.20 * df["opp_over_05_prob"]
  + 0.05 * df["rc_per90"]
  + 0.1 * df["position_yc_modifier"]
)

df["prob_yellow_card"] = raw_yc * df["prob_plays_60min"]

In [18]:
cols = [
    "name",
    "team",
    "position",
    "prob_yellow_card",
    "yc_per90",
    "stat_YC",
    "tackles_per90",
    "recent_tackles_per90",
    "opp_over_05_prob",
    "prob_plays_60min",
    "rc_per90",
]

df[cols].sort_values("prob_yellow_card", ascending=False).head(30)

,name,team,position,prob_yellow_card,yc_per90,stat_YC,tackles_per90,recent_tackles_per90,opp_over_05_prob,prob_plays_60min,rc_per90
22,Timothy Castagne,Belgium,DEF,0.305052,0.128866,0.5,0.154639,0.133333,1.000000,0.910044,0.0
128,Granit Xhaka,Switzerland,MID,0.286689,0.208333,1.0,0.052083,0.050000,0.840013,0.897707,0.0
196,Issa Diop,Morocco,DEF,0.286144,0.258398,1.0,0.000000,0.000000,0.972166,0.834743,0.0
192,Achraf Hakimi,Morocco,DEF,0.286023,0.104167,0.5,0.093750,0.066667,0.972166,0.939204,0.0
90,Patrick Berg,Norway,MID,0.280893,0.154321,0.5,0.108025,0.111111,0.944816,0.844290,0.0
28,Brandon Mechele,Belgium,DEF,0.280428,0.104167,0.5,0.052083,0.050000,1.000000,0.935736,0.0
202,Bilal El Khannouss,Morocco,MID,0.278876,0.123762,0.5,0.136139,0.125000,0.972166,0.837535,0.0
21,Maxim De Cuyper,Belgium,DEF,0.272466,0.130890,0.5,0.091623,0.116279,1.000000,0.850761,0.0
199,Neil El Aynaoui,Morocco,MID,0.267998,0.000000,0.0,0.158562,0.102389,0.972166,0.927093,0.0
39,Youri Tielemans,Belgium,MID,0.254392,0.000000,0.0,0.073684,0.067797,1.000000,0.930403,0.0


In [76]:
raw_pw = (
    0.10 * df["stat_PW"]                    
  + 0.3 * df["anytime_scorer_prob"]      
  + 0.10 * df["chance_created_per90"]     
  + 0.2 * df["stat_CC"]                  # recent form in 
  + 0.2 * df["team_score_2_prob"]        # team expected to attack heavily
  + 0.10 * df["price"] # quality proxy
)

df["prob_pen_won"] = 0.4 * sigmoid((raw_pw - 0.65) * 5) * df["prob_plays_60min"]

In [77]:
cols = [
    "name",
    "team",
    "position",
    "prob_pen_won",
    "stat_PW",
    "anytime_scorer_prob",
    "assist_expectation",
    "chance_created_per90",
    "team_score_2_prob",
    "price_rank_team_position",
    "prob_plays_60min",
]

df[cols].sort_values("prob_pen_won", ascending=False).head(30)

,name,team,position,prob_pen_won,stat_PW,anytime_scorer_prob,assist_expectation,chance_created_per90,team_score_2_prob,price_rank_team_position,prob_plays_60min
81,Michael Olise,France,MID,0.130830,0.0,0.2899,0.891520,0.766332,0.5882,0.909091,0.898315
5,Lionel Messi,Argentina,FWD,0.108530,0.0,0.0067,0.996498,0.892683,0.5348,1.000000,0.873526
68,Kylian Mbappé,France,FWD,0.096263,0.0,0.5882,0.160918,0.138322,0.5882,1.000000,0.924037
47,Harry Kane,England,FWD,0.091642,0.0,0.5556,0.158668,0.137698,0.5714,1.000000,0.924072
60,Jude Bellingham,England,MID,0.089047,0.0,0.3390,0.521955,0.452970,0.5714,0.900000,0.894069
118,Mikel Oyarzabal,Spain,FWD,0.079019,0.0,0.5348,0.183623,0.156410,0.6061,1.000000,0.909084
69,Ousmane Dembélé,France,MID,0.072955,0.0,0.4000,0.190766,0.163978,0.5882,1.000000,0.860479
114,Marc Cucurella,Spain,DEF,0.070350,0.0,0.1176,0.477420,0.406667,0.6061,0.625000,0.921290
99,Erling Haaland,Norway,FWD,0.068727,0.0,0.5051,0.164800,0.169444,0.3891,1.000000,0.835827
52,Noni Madueke,England,FWD,0.066485,1.0,0.2667,0.867778,0.753086,0.5714,0.250000,0.514746


In [21]:
raw_cs = (
    0.80 * df["team_cs_prob"]            # strongest signal
  + 0.05 * df["stat_CS"]               # tournament history
  + 0.15 * (1 - df["stat_GC"])         # fewer goals conceded is better
)

base_cs = sigmoid((raw_cs - 0.5) * 6)

df["prob_clean_sheet"] = base_cs * df["prob_plays_60min"]

df.loc[df["position"] == "FWD", "prob_clean_sheet"] = 0.0

In [22]:
cols = [
    "name",
    "team",
    "position",
    "prob_clean_sheet",
    "team_cs_prob",
    "prob_plays_60min",
    "stat_CS",
    "stat_GC",
    "minutes_rank_team_position",
    "tackles_per90",
]

df[cols].sort_values("prob_clean_sheet", ascending=False).head(30)

,name,team,position,prob_clean_sheet,team_cs_prob,prob_plays_60min,stat_CS,stat_GC,minutes_rank_team_position,tackles_per90
114,Marc Cucurella,Spain,DEF,0.483497,0.4064,0.899348,1.0,0.000,0.937500,0.022222
125,Rodri,Spain,MID,0.479499,0.4064,0.891910,1.0,0.000,1.000000,0.201342
115,Pau Cubarsí,Spain,DEF,0.479348,0.4064,0.891630,1.0,0.000,0.937500,0.022222
122,Unai Simón,Spain,GK,0.474017,0.4064,0.881713,1.0,0.000,1.000000,0.000000
112,Aymeric Laporte,Spain,DEF,0.471442,0.4064,0.876925,1.0,0.000,0.750000,0.044543
127,Pedri,Spain,MID,0.458673,0.4064,0.853173,1.0,0.000,0.909091,0.114213
116,Lamine Yamal,Spain,MID,0.450905,0.4064,0.838724,1.0,0.000,0.818182,0.111111
69,Ousmane Dembélé,France,MID,0.439098,0.4384,0.823966,0.8,0.125,0.909091,0.040323
63,Dayot Upamecano,France,DEF,0.435489,0.4384,0.889059,0.6,0.250,1.000000,0.126147
72,Mike Maignan,France,GK,0.431891,0.4384,0.881713,0.6,0.250,1.000000,0.000000


In [23]:
position_rc_modifier = {
    "GK": 0.04,
    "FWD": 0.08,
    "DEF": 0.12,
    "MID": 0.14,
}

df["position_rc_modifier"] = df["position"].map(position_rc_modifier)

raw_rc = (
    0.10 * df["position_rc_modifier"]   # strongest prior
  + 0.25 * df["rc_per90"]              # direct history
  + 0.15 * df["tackles_per90"]         # physical involvement
  + 0.25 * df["prob_yellow_card"]      # disciplinary tendency
  + 0.25 * df["opp_over_05_prob"]      # more defending -> more challenges
)

df["prob_red_card"] = 0.1 * sigmoid((raw_rc - 0.5) * 5) * df["prob_plays_60min"]

In [24]:
cols = [
    "name",
    "team",
    "position",
    "prob_red_card",
    "position_rc_modifier",
    "rc_per90",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_red_card", ascending=False).head(30)

,name,team,position,prob_red_card,position_rc_modifier,rc_per90,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
22,Timothy Castagne,Belgium,DEF,0.030344,0.12,0.0,0.154639,0.305052,1.000000,0.963988
199,Neil El Aynaoui,Morocco,MID,0.029524,0.14,0.0,0.158562,0.267998,0.972166,0.964929
192,Achraf Hakimi,Morocco,DEF,0.029179,0.12,0.0,0.093750,0.286023,0.972166,1.024389
28,Brandon Mechele,Belgium,DEF,0.029002,0.12,0.0,0.052083,0.280428,1.000000,0.975281
39,Youri Tielemans,Belgium,MID,0.028711,0.14,0.0,0.073684,0.254392,1.000000,0.991847
164,Leandro Trossard,Belgium,MID,0.027754,0.14,0.0,0.066667,0.246837,1.000000,0.988807
21,Maxim De Cuyper,Belgium,DEF,0.026728,0.12,0.0,0.091623,0.272466,1.000000,0.956984
202,Bilal El Khannouss,Morocco,MID,0.026613,0.14,0.0,0.136139,0.278876,0.972166,0.905545
194,Noussair Mazraoui,Morocco,DEF,0.026054,0.12,0.0,0.142119,0.235017,0.972166,0.917895
90,Patrick Berg,Norway,MID,0.025872,0.14,0.0,0.108025,0.280893,0.944816,0.885182


In [25]:
position_og_modifier = {
    "GK": 0.05,
    "FWD": 0.02,
    "MID": 0.04,
    "DEF": 0.08,
}

df["position_og_modifier"] = df["position"].map(position_og_modifier)

raw_og = (
    0.4 * df["position_og_modifier"]
  + 0.5 * df["opp_over_05_prob"]
  + 0.10 * (1 - df["stat_GC"])
)

df["prob_own_goal"] = 0.03 * sigmoid((raw_og - 0.5) * 5) * df["prob_plays_60min"]

In [26]:
cols = [
    "name",
    "team",
    "position",
    "prob_own_goal",
    "position_og_modifier",
    "own_goal_rate",
    "opp_over_05_prob",
    "tackles_per90",
    "stat_GC",
    "prob_plays_any",
]

df[cols].sort_values("prob_own_goal", ascending=False).head(30)

,name,team,position,prob_own_goal,position_og_modifier,own_goal_rate,opp_over_05_prob,tackles_per90,stat_GC,prob_plays_any
192,Achraf Hakimi,Morocco,DEF,0.016463,0.08,0.000000,0.972166,0.093750,0.500,1.024389
28,Brandon Mechele,Belgium,DEF,0.016451,0.08,0.000000,1.000000,0.052083,0.625,0.975281
22,Timothy Castagne,Belgium,DEF,0.015999,0.08,0.000000,1.000000,0.154639,0.625,0.963988
164,Leandro Trossard,Belgium,MID,0.015858,0.04,0.000000,1.000000,0.066667,0.500,0.988807
160,Thibaut Courtois,Belgium,GK,0.015841,0.05,0.000000,1.000000,0.000000,0.625,0.979993
39,Youri Tielemans,Belgium,MID,0.015812,0.04,0.000000,1.000000,0.073684,0.625,0.991847
191,Yassine Bounou,Morocco,GK,0.015793,0.05,0.104167,0.972166,0.000000,0.500,0.979993
194,Noussair Mazraoui,Morocco,DEF,0.015752,0.08,0.000000,0.972166,0.142119,0.250,0.917895
199,Neil El Aynaoui,Morocco,MID,0.015707,0.04,0.000000,0.972166,0.158562,0.500,0.964929
196,Issa Diop,Morocco,DEF,0.015383,0.08,0.000000,0.972166,0.000000,0.250,0.895614


In [78]:
position_pc_modifier = {
    "GK": 0.08,
    "FWD": 0.01,
    "MID": 0.05,
    "DEF": 0.14,
}

df["position_pc_modifier"] = df["position"].map(position_pc_modifier)

raw_pc = (
    0.20 * df["position_pc_modifier"]
  + 0.15 * df["penalty_conceded_rate"]
  + 0.15 * df["tackles_per90"]
  + 0.35 * df["prob_yellow_card"]
  + 0.15 * df["opp_over_05_prob"]
)

df["prob_penalty_committed"] = 0.15 * sigmoid((raw_pc - 0.5) * 5) * df["prob_plays_60min"]

In [79]:
cols = [
    "name",
    "team",
    "position",
    "prob_penalty_committed",
    "position_pc_modifier",
    "penalty_conceded_rate",
    "tackles_per90",
    "prob_yellow_card",
    "opp_over_05_prob",
    "prob_plays_any",
]

df[cols].sort_values("prob_penalty_committed", ascending=False).head(30)

,name,team,position,prob_penalty_committed,position_pc_modifier,penalty_conceded_rate,tackles_per90,prob_yellow_card,opp_over_05_prob,prob_plays_any
22,Timothy Castagne,Belgium,DEF,0.037822,0.14,0.000000,0.154639,0.305052,1.000000,0.876352
192,Achraf Hakimi,Morocco,DEF,0.036522,0.14,0.000000,0.093750,0.286023,0.972166,0.931263
28,Brandon Mechele,Belgium,DEF,0.035865,0.14,0.000000,0.052083,0.280428,1.000000,0.886620
199,Neil El Aynaoui,Morocco,MID,0.034164,0.05,0.000000,0.158562,0.267998,0.972166,0.877208
91,Kristoffer Ajer,Norway,DEF,0.033737,0.14,0.316667,0.111111,0.207140,0.944816,0.836170
21,Maxim De Cuyper,Belgium,DEF,0.033687,0.14,0.000000,0.091623,0.272466,1.000000,0.869986
39,Youri Tielemans,Belgium,MID,0.032589,0.05,0.000000,0.073684,0.254392,1.000000,0.901679
202,Bilal El Khannouss,Morocco,MID,0.031616,0.05,0.000000,0.136139,0.278876,0.972166,0.823222
164,Leandro Trossard,Belgium,MID,0.031426,0.05,0.000000,0.066667,0.246837,1.000000,0.898915
194,Noussair Mazraoui,Morocco,DEF,0.031119,0.14,0.000000,0.142119,0.235017,0.972166,0.834450


In [29]:
df["lambda_opp"] = -np.log(df["team_cs_prob"].clip(lower=0.01))

df["prob_gc_0"] = np.exp(-df["lambda_opp"])
df["prob_gc_1"] = df["lambda_opp"] * np.exp(-df["lambda_opp"])
df["prob_gc_2"] = (df["lambda_opp"] ** 2 / 2) * np.exp(-df["lambda_opp"])
df["prob_gc_3"] = (df["lambda_opp"] ** 3 / 6) * np.exp(-df["lambda_opp"])
df["prob_gc_4plus"] = 1 - (
    df["prob_gc_0"]
    + df["prob_gc_1"]
    + df["prob_gc_2"]
    + df["prob_gc_3"]
)

# RAW expected goals conceded (true intensity)
df["expected_goals_conceded"] = df["lambda_opp"]

# Normalize ONLY as auxiliary feature (separate column)
scaler = MinMaxScaler()
df["expected_goals_conceded_norm"] = scaler.fit_transform(
    df[["expected_goals_conceded"]]
)

# Player exposure to goals conceded (correct)
df["expected_player_goals_conceded"] = (
    df["lambda_opp"] * df["prob_plays_60min"]
)

# GC penalty MUST use raw λ (correct Poisson expectation)
df["expected_gc_penalty"] = (
    -(df["lambda_opp"] - 1 + np.exp(-df["lambda_opp"]))
    * df["prob_plays_60min"]
)

df.loc[df["position"].isin(["MID", "FWD"]), [
    "expected_player_goals_conceded",
    "expected_gc_penalty"
]] = 0.0

In [30]:
cols = [
    "name",
    "team",
    "position",
    "prob_plays_60min",
    "team_cs_prob",
    "lambda_opp",
    "prob_gc_0",
    "prob_gc_1",
    "prob_gc_2",
    "prob_gc_3",
    "prob_gc_4plus",
    "expected_player_goals_conceded",
    "expected_gc_penalty",
]

df[df["position"].isin(["GK", "DEF"])][cols] \
    .sort_values("expected_gc_penalty") \
    .head(30)

,name,team,position,prob_plays_60min,team_cs_prob,lambda_opp,prob_gc_0,prob_gc_1,prob_gc_2,prob_gc_3,prob_gc_4plus,expected_player_goals_conceded,expected_gc_penalty
28,Brandon Mechele,Belgium,DEF,0.935736,0.2043,1.588166,0.2043,0.324462,0.257650,0.136397,0.077191,1.486104,-0.741539
160,Thibaut Courtois,Belgium,GK,0.924142,0.2043,1.588166,0.2043,0.324462,0.257650,0.136397,0.077191,1.467690,-0.732351
22,Timothy Castagne,Belgium,DEF,0.910044,0.2043,1.588166,0.2043,0.324462,0.257650,0.136397,0.077191,1.445300,-0.721178
192,Achraf Hakimi,Morocco,DEF,0.939204,0.2115,1.553530,0.2115,0.328572,0.255223,0.132166,0.072540,1.459081,-0.718519
191,Yassine Bounou,Morocco,GK,0.924142,0.2115,1.553530,0.2115,0.328572,0.255223,0.132166,0.072540,1.435682,-0.706996
21,Maxim De Cuyper,Belgium,DEF,0.850761,0.2043,1.588166,0.2043,0.324462,0.257650,0.136397,0.077191,1.351150,-0.674199
194,Noussair Mazraoui,Morocco,DEF,0.854722,0.2115,1.553530,0.2115,0.328572,0.255223,0.132166,0.072540,1.327836,-0.653888
196,Issa Diop,Morocco,DEF,0.834743,0.2115,1.553530,0.2115,0.328572,0.255223,0.132166,0.072540,1.296799,-0.638604
144,Gregor Kobel,Switzerland,GK,0.906785,0.2434,1.413049,0.2434,0.343936,0.242999,0.114457,0.055208,1.281332,-0.595258
133,Manuel Akanji,Switzerland,DEF,0.896251,0.2434,1.413049,0.2434,0.343936,0.242999,0.114457,0.055208,1.266447,-0.588343


In [31]:
raw_save = (
    0.35 * df["expected_goals_conceded_norm"]  # opportunity
  + 0.40 * df["saves_per90"]             # ability
  + 0.05 * df["recent_saves_per90"]      # recent form
  + 0.20 * df["price"]                   # overall GK quality
)

# Expected saves (λ)
df["expected_saves"] = (
    sigmoid((raw_save - 0.5) * 6) * 4
) * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "expected_saves"] = 0.0

In [32]:
cols = [
    "name",
    "team",
    "expected_saves",
    "expected_goals_conceded",
    "saves_per90",
    "recent_saves_per90",
    "price",
    "expected_goals_conceded_norm",
]

df[df["position"] == "GK"][cols] \
    .sort_values("expected_saves", ascending=False) \
    .head(30)

,name,team,expected_saves,expected_goals_conceded,saves_per90,recent_saves_per90,price,expected_goals_conceded_norm
160,Thibaut Courtois,Belgium,2.847462,1.588166,0.3750,0.300000,0.933333,1.000000
144,Gregor Kobel,Switzerland,2.811876,1.413049,0.6000,0.660000,0.800000,0.780997
191,Yassine Bounou,Morocco,2.604399,1.553530,0.3375,0.300000,0.800000,0.956684
103,Ørjan Nyland,Norway,2.201627,1.518684,0.5000,0.800000,0.466667,0.913105
53,Jordan Pickford,England,1.570326,1.076459,0.3600,0.400000,0.866667,0.360054
72,Mike Maignan,France,1.080420,0.824624,0.3200,0.400000,1.000000,0.045105
122,Unai Simón,Spain,1.055890,0.900417,0.2400,0.266667,1.000000,0.139893
104,Egil Selvik,Norway,0.850405,1.518684,1.0000,1.000000,0.200000,0.913105
10,Emiliano Martínez,Argentina,0.739488,0.788557,0.1500,0.180000,1.000000,0.000000
36,Senne Lammens,Belgium,0.244465,1.588166,0.0000,0.000000,0.733333,1.000000


In [33]:
raw_pen_save = (
    0.55 * df["price"]                    
  + 0.30 * df["expected_goals_conceded_norm"]
  + 0.15 * df["stat_PS"]
)

# Strong compression because penalty saves are exceptionally rare
df["prob_penalty_save"] = sigmoid((raw_pen_save - 0.5) * 3) / 20 * df["prob_plays_60min"]

df.loc[df["position"] != "GK", "prob_penalty_save"] = 0.0

In [34]:
cols = [
    "name",
    "team",
    "prob_penalty_save",
    "price",
    "expected_goals_conceded",
    "opp_over_05_prob",
    "prob_plays_60min",
    "stat_PS",
]

df[df["position"] == "GK"][cols] \
    .sort_values("prob_penalty_save", ascending=False) \
    .head(30)

,name,team,prob_penalty_save,price,expected_goals_conceded,opp_over_05_prob,prob_plays_60min,stat_PS
160,Thibaut Courtois,Belgium,0.033228,0.933333,1.588166,1.000000,0.924142,0.0
191,Yassine Bounou,Morocco,0.030680,0.800000,1.553530,0.972166,0.924142,0.0
72,Mike Maignan,France,0.028871,1.000000,0.824624,0.067909,0.881713,1.0
144,Gregor Kobel,Switzerland,0.028465,0.800000,1.413049,0.840013,0.906785,0.0
122,Unai Simón,Spain,0.025065,1.000000,0.900417,0.185855,0.881713,0.0
10,Emiliano Martínez,Argentina,0.024833,1.000000,0.788557,0.000000,0.924142,0.0
53,Jordan Pickford,England,0.024828,0.866667,1.076459,0.444133,0.881713,0.0
103,Ørjan Nyland,Norway,0.024350,0.466667,1.518684,0.944816,0.770299,1.0
104,Egil Selvik,Norway,0.005085,0.200000,1.518684,0.944816,0.245779,0.0
36,Senne Lammens,Belgium,0.004000,0.733333,1.588166,1.000000,0.123467,0.0


In [86]:
raw_tackles = (
    0.50 * df["tackles_per90"]
  + 0.15 * df["recent_tackles_per90"]
  + 0.15 * df["expected_goals_conceded_norm"]
  + 0.20 * df["match_over_25_prob"]
)

df["lambda_tackles"] = -np.log(
    (1 - raw_tackles).clip(lower=0.001)
)

df["expected_tackles"] = (
    df["lambda_tackles"].clip(lower=0)
    * df["prob_plays_60min"] * 3
)

df.loc[df["position"] != "MID", "expected_tackles"] = 0.0

In [87]:
cols = [
    "name",
    "team",
    "expected_tackles",
    "tackles_per90",
    "recent_tackles_per90",
    "expected_goals_conceded",
    "match_over_25_prob",
    "price",
    "prob_plays_60min",
]

df[df["position"] == "MID"][cols] \
    .sort_values("expected_tackles", ascending=False) \
    .head(30)

,name,team,expected_tackles,tackles_per90,recent_tackles_per90,expected_goals_conceded,match_over_25_prob,price,prob_plays_60min
199,Neil El Aynaoui,Morocco,1.151610,0.158562,0.102389,1.553530,0.4930,0.169811,0.934919
202,Bilal El Khannouss,Morocco,1.033554,0.136139,0.125000,1.553530,0.4930,0.283019,0.863735
39,Youri Tielemans,Belgium,1.008921,0.073684,0.067797,1.588166,0.5211,0.264151,0.938262
90,Patrick Berg,Norway,0.974213,0.108025,0.111111,1.518684,0.5365,0.169811,0.858514
164,Leandro Trossard,Belgium,0.955052,0.066667,0.037037,1.588166,0.5211,0.358491,0.917758
87,Sander Berge,Norway,0.887551,0.083333,0.111111,1.518684,0.5365,0.000000,0.820906
177,Martin Ødegaard,Norway,0.862094,0.085470,0.083333,1.518684,0.5365,0.566038,0.807291
58,Elliot Anderson,England,0.854991,0.169492,0.180723,1.076459,0.5365,0.339623,0.893262
35,Charles De Ketelaere,Belgium,0.852186,0.114068,0.101523,1.588166,0.5211,0.169811,0.718705
125,Rodri,Spain,0.847423,0.201342,0.259259,0.900417,0.5211,0.528302,0.918425


In [66]:
raw_cc = (
    0.25 * df["recent_cc_per90"]
  + 0.25 * df["stat_CC"]
  + 0.25 * df["match_over_25_prob"]
  + 0.15 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_cc"] = -np.log(
    (1 - raw_cc).clip(lower=0.001)
)

df["expected_chances_created"] = (
    df["lambda_cc"].clip(lower=0)
    * df["prob_plays_60min"] * 2
)

df.loc[df["position"] != "MID", "expected_chances_created"] = 0.0

In [67]:
cols = [
    "name",
    "team",
    "position",
    "expected_chances_created",
    "cc_per90",
    "recent_cc_per90",
    "stat_CC",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_chances_created", ascending=False).head(30)

,name,team,position,expected_chances_created,cc_per90,recent_cc_per90,stat_CC,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
81,Michael Olise,France,MID,1.397614,0.766332,0.300000,0.833333,0.4930,0.2899,0.905660,0.898315
60,Jude Bellingham,England,MID,1.188218,0.452970,0.430279,0.500000,0.5365,0.3390,0.679245,0.894069
164,Leandro Trossard,Belgium,MID,1.010613,0.406667,0.400000,0.500000,0.5211,0.2151,0.358491,0.917758
200,Brahim Díaz,Morocco,MID,0.963732,0.628866,0.301255,0.666667,0.4930,0.2222,0.320755,0.855532
59,Declan Rice,England,MID,0.860982,0.536657,0.402235,0.500000,0.5365,0.1639,0.433962,0.771447
90,Patrick Berg,Norway,MID,0.768845,0.564815,0.266667,0.500000,0.5365,0.1212,0.169811,0.858514
48,Bukayo Saka,England,MID,0.728125,0.635417,0.483221,0.333333,0.5365,0.2985,0.905660,0.567338
201,Azzedine Ounahi,Morocco,MID,0.715941,0.527378,0.373057,0.500000,0.4930,0.1818,0.283019,0.707478
69,Ousmane Dembélé,France,MID,0.676197,0.163978,0.000000,0.166667,0.4930,0.4000,1.000000,0.860479
126,Dani Olmo,Spain,MID,0.614613,0.476562,0.193548,0.333333,0.5211,0.2985,0.566038,0.680524


In [64]:
raw_sot = (
    0.35 * df["sot_per90"]
  + 0.15 * df["stat_GS"]
  + 0.20 * df["match_over_25_prob"]
  + 0.20 * df["anytime_scorer_prob"]
  + 0.10 * df["price"]
)

df["lambda_sot"] = -np.log(
    (1 - raw_sot).clip(lower=0.001)
)

df["expected_shots_on_target"] = (
    df["lambda_sot"].clip(lower=0)
    * df["prob_plays_60min"] * 2
)

df.loc[df["position"] != "FWD", "expected_shots_on_target"] = 0.0

In [65]:
cols = [
    "name",
    "team",
    "position",
    "expected_shots_on_target",
    "sot_per90",
    "goal_rate",
    "stat_GS",
    "match_over_25_prob",
    "anytime_scorer_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("expected_shots_on_target", ascending=False).head(30)

,name,team,position,expected_shots_on_target,sot_per90,goal_rate,stat_GS,match_over_25_prob,anytime_scorer_prob,price,prob_plays_60min
68,Kylian Mbappé,France,FWD,2.012090,0.616780,0.813492,0.875,0.4930,0.5882,1.000000,0.924037
99,Erling Haaland,Norway,FWD,1.645129,0.533333,0.996528,0.875,0.5365,0.5051,1.000000,0.835827
47,Harry Kane,England,FWD,1.506112,0.361174,0.694131,0.750,0.5365,0.5556,1.000000,0.924072
5,Lionel Messi,Argentina,FWD,1.453149,0.663415,1.000000,1.000,0.4444,0.0067,0.923077,0.873526
118,Mikel Oyarzabal,Spain,FWD,1.183668,0.369231,0.525641,0.500,0.5211,0.5348,0.630769,0.909084
203,Soufiane Rahimi,Morocco,FWD,0.627458,0.496124,0.794574,0.250,0.4930,0.2667,0.246154,0.639570
31,Romelu Lukaku,Belgium,FWD,0.585986,0.240000,0.768750,0.375,0.5211,0.3279,0.523077,0.651128
141,Breel Embolo,Switzerland,FWD,0.473331,0.147465,0.236175,0.250,0.4444,0.0067,0.538462,0.891388
143,Dan Ndoye,Switzerland,FWD,0.422081,0.298137,0.159161,0.125,0.4444,0.0067,0.430769,0.712386
6,Julián Alvarez,Argentina,FWD,0.368940,0.166090,0.000000,0.000,0.4444,0.0067,0.707692,0.745831


In [41]:
df["expected_qualification_points"] = df["team_qualify_prob"] * 0

In [75]:
cols = [
    "name",
    "team",
    "position",
    "expected_qualification_points",
    "team_qualify_prob",
    "price",
    "prob_plays_60min",
]

df[cols].sort_values("team_qualify_prob", ascending=False).groupby("team").head(3).head(30)

,name,team,position,expected_qualification_points,team_qualify_prob,price,prob_plays_60min
75,Adrien Rabiot,France,MID,0.0,0.787402,0.320755,0.754157
74,N'Golo Kanté,France,MID,0.0,0.787402,0.075472,0.098694
73,Brice Samba,France,GK,0.0,0.787402,0.666667,0.122389
114,Marc Cucurella,Spain,DEF,0.0,0.781250,0.640000,0.921290
189,Nico Williams,Spain,MID,0.0,0.781250,0.584906,0.131628
187,Fabián Ruiz,Spain,MID,0.0,0.781250,0.396226,0.238841
9,José Manuel López,Argentina,FWD,0.0,0.769231,0.200000,0.091040
12,Juan Musso,Argentina,GK,0.0,0.769231,0.533333,0.122389
5,Lionel Messi,Argentina,FWD,0.0,0.769231,0.923077,0.873526
42,Marc Guéhi,England,DEF,0.0,0.694444,0.640000,0.876983


In [43]:
df.to_csv(OUT_CSV, index=False)

print(f"\nSaved {OUT_CSV} {df.shape[0]} rows x {df.shape[1]} cols")


Saved fantasy_enriched.csv 206 rows x 247 cols


In [ ]:
BUDGET = 105.0
MAX_PER_COUNTRY = 5
SQUAD_SIZE = 15
XI_SIZE = 11
BENCH_WEIGHT = 0.85

def compute_total_expected_points(df):
    df = df.copy()

    cs_value = {"GK": 5, "DEF": 5, "MID": 1, "FWD": 0}
    goal_value = {"GK": 9, "DEF": 7, "MID": 6, "FWD": 5}

    df["goal_pts_value"] = df["position"].map(goal_value)
    df["cs_pts_value"] = df["position"].map(cs_value)

    # appearance: 1pt for playing any, +1 if 60+
    e_appearance = df["prob_plays_any"] * 1 + df["prob_plays_60min"] * 1

    # goal
    e_goal = df["prob_scores"] * df["goal_pts_value"]

    # assist
    e_assist = df["expected_assists"] * 3

    # clean sheet: gated by 60min, value depends on position
    e_cs = df["prob_clean_sheet"] * df["cs_pts_value"]

    # goals conceded penalty: GK and DEF only
    e_gc = df["expected_gc_penalty"].copy()
    e_gc[~df["position"].isin(["GK", "DEF"])] = 0.0

    # saves bonus: GK only, every 3 saves = +1
    e_saves = (df["expected_saves"] / 3).copy()
    e_saves[df["position"] != "GK"] = 0.0

    # penalty save: GK only
    e_pen_save = (df["prob_penalty_save"] * 3).copy()
    e_pen_save[df["position"] != "GK"] = 0.0

    # tackles bonus: MID only, every 3 tackles = +1
    e_tackles = (df["expected_tackles"] / 3).copy()
    e_tackles[df["position"] != "MID"] = 0.0

    # big chances created: MID only, every 2 = +1
    e_cc = (df["expected_chances_created"] / 2).copy()
    e_cc[df["position"] != "MID"] = 0.0

    # shots on target: FWD only, every 2 = +1
    e_sot = (df["expected_shots_on_target"] / 2).copy()
    e_sot[df["position"] != "FWD"] = 0.0

    # yellow card
    e_yc = df["prob_yellow_card"] * -1

    # red card
    e_rc = df["prob_red_card"] * -2

    # own goal
    e_og = df["prob_own_goal"] * -2

    # penalty won
    e_pw = df["prob_pen_won"] * 2

    # penalty committed
    e_pc = df["prob_penalty_committed"] * -1

    # qualification bonus: +2 per player in XI who advances, requires playing
    # captain's quali bonus is NOT doubled per rules
    e_quali = df["expected_qualification_points"] * df["prob_plays_any"]

    # scouting bonus: +2 if differential==1 AND player scores >4 base pts
    # model: use Poisson confidence interval on expected base points
    # base pts = everything except scouting and captain multiplier
    base_ep = (
        e_appearance + e_goal + e_assist + e_cs + e_gc +
        e_saves + e_pen_save + e_tackles + e_cc + e_sot +
        e_yc + e_rc + e_og + e_pw + e_pc + e_quali
    )

    # Poisson 90% CI: lower = ppf(0.05, mu), upper = ppf(0.95, mu)
    mu = base_ep.clip(lower=0.01)
    lower_ci = pd.Series(poisson.ppf(0.05, mu), index=df.index)
    upper_ci = pd.Series(poisson.ppf(0.95, mu), index=df.index)

    # if lower CI > 4: full +2 expected
    # if upper CI < 4: 0
    # if CI straddles 4: scale by P(X>4 | mu) * 2
    p_exceeds_4 = pd.Series(1 - poisson.cdf(4, mu), index=df.index)

    scouting_ep = pd.Series(0.0, index=df.index)
    eligible = df["differential"] == 1
    full_bonus = eligible & (lower_ci > 4)
    no_bonus = eligible & (upper_ci < 4)
    partial = eligible & ~full_bonus & ~no_bonus

    scouting_ep[full_bonus] = 2.0
    scouting_ep[no_bonus] = 0.0
    scouting_ep[partial] = p_exceeds_4[partial] * 2.0

    df["e_appearance"] = e_appearance
    df["e_goal"] = e_goal
    df["e_assist"] = e_assist
    df["e_cs"] = e_cs
    df["e_gc"] = e_gc
    df["e_saves"] = e_saves
    df["e_pen_save"] = e_pen_save
    df["e_tackles"] = e_tackles
    df["e_cc"] = e_cc
    df["e_sot"] = e_sot
    df["e_yc"] = e_yc
    df["e_rc"] = e_rc
    df["e_og"] = e_og
    df["e_pw"] = e_pw
    df["e_pc"] = e_pc
    df["e_quali"] = e_quali
    df["e_scouting"] = scouting_ep

    e_elim_risk = -1 * (1 - df["team_qualify_prob"])
    df["e_elim_risk"] = e_elim_risk

    df["expected_points"] = base_ep + scouting_ep + e_elim_risk

    return df

In [45]:

def optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):
    idx = df.index.tolist()

    x = LpVariable.dicts("squad", idx, cat="Binary")
    s = LpVariable.dicts("xi", idx, cat="Binary")
    b = LpVariable.dicts("bench", idx, cat="Binary")    
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("fantasy", LpMaximize)

    
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
        for i in idx
    )

    # squad = 15, xi = 11, bench = 4
    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    # budget
    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    # squad composition: 2 GK, 5 DEF, 5 MID, 3 FWD
    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    # country limit
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country


    # max 2 GK+DEF combined from the same country
    for country in df["team"].unique():
        gkdef_idx = df[(df["team"] == country) & (df["position"].isin(["GK", "DEF"]))].index.tolist()
        model += lpSum(x[i] for i in gkdef_idx) <= 2

    # XI formation: 1 GK starts, valid outfield formation
    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    # captain: exactly 1, must be in XI
    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad = [i for i in idx if value(x[i]) > 0.5]
    in_xi = [i for i in idx if value(s[i]) > 0.5]
    on_bench = [i for i in idx if value(b[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "squad": df.loc[in_squad].copy(),
        "xi": df.loc[in_xi].copy(),
        "bench": df.loc[on_bench].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
        "cost": df.loc[in_squad, "price_raw"].sum(),
    }

In [46]:
def optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY):
    # best possible 11 ignoring bench constraint, just 11 players
    idx = df.index.tolist()

    s = LpVariable.dicts("xi", idx, cat="Binary")
    cap = LpVariable.dicts("cap", idx, cat="Binary")

    model = LpProblem("ideal_xi", LpMaximize)

    # optimize_ideal_xi objective — NO bench term
    model += lpSum(
        df.loc[i, "expected_points"] * s[i]
        + df.loc[i, "expected_points"] * cap[i]
        for i in idx
    )

    model += lpSum(s[i] for i in idx) == 11

    # no budget constraint for ideal XI comparison
    # country limit still applies
    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(s[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        return None

    in_xi = [i for i in idx if value(s[i]) > 0.5]
    captain = next(i for i in idx if value(cap[i]) > 0.5)

    return {
        "xi": df.loc[in_xi].copy(),
        "captain": captain,
        "total_ep": value(model.objective),
    }


def print_team(result, ideal=None):
    pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
    cap = result["captain"]

    xi = result["xi"].copy()
    xi["_ord"] = xi["position"].map(pos_order)
    xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])

    bench = result["bench"].copy()
    bench["_ord"] = bench["position"].map(pos_order)
    bench = bench.sort_values(["_ord", "expected_points"], ascending=[True, False])

    print(f"\nExpected points: {result['total_ep']:.2f}   Cost: ${result['cost']:.1f}M\n")

    print("Starting XI")
    for i, row in xi.iterrows():
        tag = " [C]" if i == cap else ""
        scout = " [SCOUT]" if row.get("differential", 0) == 1 else ""
        print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{scout}")

    print("\nBench")
    for rank, (i, row) in enumerate(bench.iterrows(), 1):
        print(f"  [{rank}] {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}")

    print(f"\nCountry breakdown: {dict(result['squad']['team'].value_counts())}")

    if ideal:
        ideal_xi = ideal["xi"].copy()
        ideal_xi["_ord"] = ideal_xi["position"].map(pos_order)
        ideal_xi = ideal_xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        ideal_cap = ideal["captain"]

        print(f"\nIdeal XI (no budget constraint, 11 only, EP: {ideal['total_ep']:.2f})")
        for i, row in ideal_xi.iterrows():
            tag = " [C]" if i == ideal_cap else ""
            in_squad = i in result["squad"].index
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")
            
if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)

    df = compute_total_expected_points(df)

    result = optimize_squad(df, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)

    print_team(result, ideal)


Expected points: 99.03   Cost: $104.7M

Starting XI
  GK   Mike Maignan                 France                 $5.0  EP:4.01
  DEF  Marc Cucurella               Spain                  $5.1  EP:5.97
  DEF  Aymeric Laporte              Spain                  $5.5  EP:4.97
  DEF  Jules Koundé                 France                 $5.4  EP:4.70
  MID  Michael Olise                France                 $9.5  EP:8.64
  MID  Jude Bellingham              England                $8.3  EP:7.37
  MID  Ousmane Dembélé              France                 $10.0  EP:7.17
  MID  Brahim Díaz                  Morocco                $6.4  EP:6.60 [SCOUT]
  FWD  Kylian Mbappé                France                 $10.5  EP:9.46 [C]
  FWD  Harry Kane                   England                $10.5  EP:8.39
  FWD  Mikel Oyarzabal              Spain                  $8.1  EP:7.60

Bench
  [1] GK   Emiliano Martínez            Argentina              $5.0  EP:3.69
  [2] DEF  Lisandro Martínez            Argen

In [47]:
def optimize_with_transfers(df, current_team_names, free_transfers=4, budget=BUDGET, max_per_country=MAX_PER_COUNTRY):

    current_idx = set()
    unmatched = []
    for name in current_team_names:
        match = df[df["name"].str.lower().str.strip() == name.lower().strip()]
        if len(match) == 0:
            match = df[df["name"].str.lower().str.contains(name.lower().strip(), na=False)]
        if len(match) > 0:
            current_idx.add(match.index[0])
        else:
            unmatched.append(name)

  
    if unmatched:
        print(f"Eliminated players: {unmatched}")

    forced_transfers = len(unmatched)

    idx = df.index.tolist()

    x         = LpVariable.dicts("squad", idx, cat="Binary")
    s         = LpVariable.dicts("xi",    idx, cat="Binary")
    b         = LpVariable.dicts("bench", idx, cat="Binary")
    cap       = LpVariable.dicts("cap",   idx, cat="Binary")
    transfer_out = LpVariable.dicts("out", idx, cat="Binary")

    model = LpProblem("fantasy_transfers", LpMaximize)

    transfers_made = lpSum(transfer_out[i] for i in idx)   # voluntary drops only

    n_extra = LpVariable("n_extra", lowBound=0)

    # CHANGE 2: total transfers = voluntary + forced. Penalty fires when
    # (voluntary + forced) > free_transfers, not just voluntary > free_transfers.
    model += n_extra >= transfers_made + forced_transfers - free_transfers

    model += (
        lpSum(
            df.loc[i, "expected_points"] * s[i]
            + df.loc[i, "expected_points"] * cap[i]
            + df.loc[i, "expected_points"] * BENCH_WEIGHT * b[i]
            for i in idx
        )
        - 3 * n_extra
    )

    for i in idx:
        if i in current_idx:
            model += transfer_out[i] == 1 - x[i]
        else:
            model += transfer_out[i] == 0

    model += lpSum(x[i] for i in idx) == 15
    model += lpSum(s[i] for i in idx) == 11
    model += lpSum(b[i] for i in idx) == 4

    for i in idx:
        model += s[i] + b[i] == x[i]

    model += lpSum(df.loc[i, "price_raw"] * x[i] for i in idx) <= budget

    for pos, count in [("GK", 2), ("DEF", 5), ("MID", 5), ("FWD", 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(x[i] for i in p_idx) == count

    for country in df["team"].unique():
        c_idx = df[df["team"] == country].index.tolist()
        model += lpSum(x[i] for i in c_idx) <= max_per_country

    gk_idx = df[df["position"] == "GK"].index.tolist()
    model += lpSum(s[i] for i in gk_idx) == 1
    model += lpSum(b[i] for i in gk_idx) == 1

    for pos, lo, hi in [("DEF", 3, 5), ("MID", 3, 5), ("FWD", 1, 3)]:
        p_idx = df[df["position"] == pos].index.tolist()
        model += lpSum(s[i] for i in p_idx) >= lo
        model += lpSum(s[i] for i in p_idx) <= hi

    model += lpSum(cap[i] for i in idx) == 1
    for i in idx:
        model += cap[i] <= s[i]

    model.solve(PULP_CBC_CMD(msg=0))

    if LpStatus[model.status] != "Optimal":
        print(f"Solver status: {LpStatus[model.status]}")
        return None

    in_squad  = [i for i in idx if value(x[i]) > 0.5]
    in_xi     = [i for i in idx if value(s[i]) > 0.5]
    on_bench  = [i for i in idx if value(b[i]) > 0.5]
    captain   = next(i for i in idx if value(cap[i]) > 0.5)

    new_squad_idx  = set(in_squad)
    voluntary_out  = [i for i in current_idx if i not in new_squad_idx]
    transfers_in   = [i for i in new_squad_idx if i not in current_idx]

    # CHANGE 3: total = voluntary drops of matched players + forced drops of eliminated.
    n_transfers = len(voluntary_out) + forced_transfers
    penalty     = max(0, n_transfers - free_transfers) * 3

    return {
        "squad":         df.loc[in_squad].copy(),
        "xi":            df.loc[in_xi].copy(),
        "bench":         df.loc[on_bench].copy(),
        "captain":       captain,
        "total_ep":      value(model.objective),
        "cost":          df.loc[in_squad, "price_raw"].sum(),
        "transfers_out": df.loc[voluntary_out, "name"].tolist() + unmatched,  # eliminated shown in OUT
        "transfers_in":  df.loc[transfers_in, "name"].tolist(),
        "n_transfers":   n_transfers,
        "penalty":       penalty,
        "forced_out":    unmatched,   # separate field for clarity
    }

def print_transfers(result):
    if result is None:
        print("No solution.")
        return

    print(f"\nTransfers made: {result['n_transfers']} ({result['n_transfers'] - 4} penalised at -3 each)" if result['n_transfers'] > 4 else f"\nTransfers made: {result['n_transfers']} (all free)")
    print(f"Point penalty: -{result['penalty']}")

    if result["transfers_out"]:
        print("\nOUT:")
        for name in result["transfers_out"]:
            print(f"  {name}")
        print("IN:")
        for name in result["transfers_in"]:
            print(f"  {name}")

    print_team(result)


if __name__ == "__main__":
    df = pd.read_csv("fantasy_enriched.csv")
    df = df[df["status"] == "playing"].copy()
    df = df.reset_index(drop=True)
    df = compute_total_expected_points(df)

    current_team = [
        "Emiliano Martínez", "Lisandro Martínez", "Achraf Hakimi", "Nico O'Reilly", "Marc Cucurella",
        "Ousmane Dembélé", "Michael Olise", "Vinícius Júnior", "Christian Pulisic", "Kylian Mbappé", "Mikel Oyarzabal",
        "Camilo Vargas", "Facundo Medina", "Lionel Messi", "Johan Manzambi"
    ]

    result = optimize_with_transfers(df, current_team, free_transfers=4, budget=BUDGET, max_per_country=MAX_PER_COUNTRY)
    ideal = optimize_ideal_xi(df, max_per_country=MAX_PER_COUNTRY)
    print_transfers(result)

    print("\nIdeal XI for comparison:")
    if ideal:
        pos_order = {"GK": 0, "DEF": 1, "MID": 2, "FWD": 3}
        xi = ideal["xi"].copy()
        xi["_ord"] = xi["position"].map(pos_order)
        xi = xi.sort_values(["_ord", "expected_points"], ascending=[True, False])
        cap = ideal["captain"]
        for i, row in xi.iterrows():
            tag = " [C]" if i == cap else ""
            in_squad = i in result["squad"].index if result else False
            flag = "" if in_squad else " [NOT IN SQUAD]"
            print(f"  {row['position']:3}  {row['name']:<28} {row['team']:<22} ${row['price_raw']:.1f}  EP:{row['expected_points']:.2f}{tag}{flag}")

Eliminated players: ['Vinícius Júnior', 'Christian Pulisic', 'Camilo Vargas', 'Johan Manzambi']

Transfers made: 4 (all free)
Point penalty: -0

OUT:
  Vinícius Júnior
  Christian Pulisic
  Camilo Vargas
  Johan Manzambi
IN:
  Brahim Díaz
  Unai Simón
  Jude Bellingham
  Dani Olmo

Expected points: 93.31   Cost: $104.9M

Starting XI
  GK   Unai Simón                   Spain                  $5.0  EP:4.12
  DEF  Marc Cucurella               Spain                  $5.1  EP:5.97
  DEF  Lisandro Martínez            Argentina              $4.6  EP:4.01
  DEF  Nico O'Reilly                England                $4.7  EP:3.58
  MID  Michael Olise                France                 $9.5  EP:8.64
  MID  Jude Bellingham              England                $8.3  EP:7.37
  MID  Ousmane Dembélé              France                 $10.0  EP:7.17
  MID  Brahim Díaz                  Morocco                $6.4  EP:6.60 [SCOUT]
  FWD  Kylian Mbappé                France                 $10.5  EP:9.4

In [48]:
for pos in ["GK", "DEF", "MID", "FWD"]:
    print(f"\n--- {pos} ---")
    print(df[df["position"] == pos].nlargest(10, "expected_points")[["name", "team", "price_raw", "expected_points"]].to_string(index=False))


--- GK ---
             name        team  price_raw  expected_points
       Unai Simón       Spain        5.0         4.116899
     Mike Maignan      France        5.0         4.009207
Emiliano Martínez   Argentina        5.0         3.687529
     Gregor Kobel Switzerland        4.7         2.981540
  Jordan Pickford     England        4.8         2.952171
   Yassine Bounou     Morocco        4.7         2.143223
 Thibaut Courtois     Belgium        4.9         2.067522
     Ørjan Nyland      Norway        4.2         1.921731
       David Raya       Spain        5.0         0.503959
   Gerónimo Rulli   Argentina        4.5         0.501101

--- DEF ---
             name      team  price_raw  expected_points
   Marc Cucurella     Spain        5.1         5.973669
  Aymeric Laporte     Spain        5.5         4.973861
     Jules Koundé    France        5.4         4.697733
  Dayot Upamecano    France        5.3         4.693018
      Pau Cubarsí     Spain        5.0         4.517173
 

In [49]:
player1 = "Brahim Díaz"
player2 = "Achraf Hakimi"

cols = [
    "name", "team", "position", "price_raw", "expected_points",
    "e_appearance", "e_goal", "e_assist", "e_cs", "e_gc",
    "e_saves", "e_pen_save", "e_tackles", "e_cc", "e_sot",
    "e_yc", "e_rc", "e_og", "e_pw", "e_pc",
    "e_quali", "e_scouting", "e_elim_risk"
]

comparison = df.loc[
    df["name"].isin([player1, player2]),
    cols
].set_index("name").T

print(comparison)

name            Achraf Hakimi Brahim Díaz
team                  Morocco     Morocco
position                  DEF         MID
price_raw                 6.0         6.4
expected_points      3.347474    6.603813
e_appearance         1.963593    1.770928
e_goal               1.379927    1.450906
e_assist             0.813628    2.067007
e_cs                 0.917968    0.162358
e_gc                -0.718519         0.0
e_saves                   0.0         0.0
e_pen_save                0.0         0.0
e_tackles                 0.0     0.03866
e_cc                      0.0    0.628866
e_sot                     0.0         0.0
e_yc                -0.286023   -0.215199
e_rc                -0.058358   -0.047444
e_og                -0.032926   -0.028143
e_pw                 0.054183    0.060928
e_pc                -0.019333    -0.01406
e_quali                   0.0         0.0
e_scouting                0.0    1.395673
e_elim_risk         -0.666667   -0.666667


In [50]:
player1 = "Emiliano Martínez"
player2 = "Erling Haaland"

cols = [
    "name", "team", "position", "price_raw", "expected_points",
    "e_appearance", "e_goal", "e_assist", "e_cs", "e_gc",
    "e_saves", "e_pen_save", "e_tackles", "e_cc", "e_sot",
    "e_yc", "e_rc", "e_og", "e_pw", "e_pc",
    "e_quali", "e_scouting"
]

comparison = df.loc[
    df["name"].isin([player1, player2]),
    cols
].set_index("name").T

print(comparison)

name            Emiliano Martínez Erling Haaland
team                    Argentina         Norway
position                       GK            FWD
price_raw                     5.0           10.5
expected_points          3.687529       6.785128
e_appearance             1.904135       1.773756
e_goal                        0.0       4.451413
e_assist                 0.023183       0.800323
e_cs                     1.897918            0.0
e_gc                    -0.224619            0.0
e_saves                  0.246496            0.0
e_pen_save               0.074499            0.0
e_tackles                     0.0            0.0
e_cc                          0.0            0.0
e_sot                         0.0       0.533333
e_yc                    -0.013862      -0.182603
e_rc                    -0.014512      -0.042248
e_og                    -0.005469      -0.025554
e_pw                     0.036703       0.067041
e_pc                    -0.006173      -0.012275
e_quali             